In [2]:
import pandas as pd
import numpy as np
import os
import glob
import warnings

# Tắt các cảnh báo không cần thiết để notebook gọn gàng
warnings.filterwarnings('ignore')

input_dir = r"D:\detection_ddos\data\CSV-01-12\training\org"
output_dir = r"D:\detection_ddos\data\CSV-01-12\training\cleaned"

# 2. Tạo thư mục output 
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Đã tạo thư mục đầu ra: {output_dir}")
else:
    print(f"Thư mục đầu ra đã sẵn sàng: {output_dir}")

print("Import thư viện và thiết lập đường dẫn thành công!")

Thư mục đầu ra đã sẵn sàng: D:\detection_ddos\data\CSV-01-12\training\cleaned
Import thư viện và thiết lập đường dẫn thành công!


In [3]:
import pandas as pd
import numpy as np
import os
import glob

input_dir = r"D:\detection_ddos\data\CSV-01-12\training\org"
output_dir = r"D:\detection_ddos\data\CSV-01-12\training\cleaned"
chunk_size = 500000

csv_files = glob.glob(os.path.join(input_dir, "*.csv"))


for file in csv_files:
    file_name = os.path.basename(file)
    print(f"\n{'='*50}")
    print(f"--- ĐANG XỬ LÝ FILE: {file_name} ---")

    #  tìm vị trí nhãn 0
    benign_indices = []
    total_rows = 0

    for chunk in pd.read_csv(file, chunksize=chunk_size, low_memory=False):
        chunk.columns = chunk.columns.str.strip()
        if 'Label' not in chunk.columns:
            continue
            
        # Nhận diện nhãn 0
        is_benign = chunk['Label'].astype(str).str.strip().str.upper() == 'BENIGN'
        idx_0 = chunk[is_benign].index.tolist()
        benign_indices.extend(idx_0)
        total_rows += len(chunk)

    n_benign = len(benign_indices)
    print(f"Hoàn tất Pass 1. Tổng số dòng: {total_rows} | Số nhãn 0 (BENIGN): {n_benign}")

    if n_benign == 0:
        print("-> Không có nhãn 0 trong file này.")
        continue

    # lấy nhãn theo tỷ lệ
    target_total_rows = int(n_benign / 0.3)
    

    target_ddos_rows = target_total_rows - n_benign
    print(f"Số lượng nhãn 1 (DDoS) cần lấy xung quanh nhãn 0: {target_ddos_rows}")

    indices_to_keep = set(benign_indices)
    window = 1
    
    # dynamic windows
    while len(indices_to_keep) < target_total_rows and window < total_rows:
        for idx in benign_indices:
            if idx - window >= 0:
                indices_to_keep.add(idx - window)
            if idx + window < total_rows:
                indices_to_keep.add(idx + window)
            
            if len(indices_to_keep) >= target_total_rows:
                break
        window += 1 

    final_indices_to_extract = sorted(list(indices_to_keep))

    # lấy dữ liệu
    print("Rút trích dữ liệu từ file...")
    file_sampled_chunks = []
    
    for i, chunk in enumerate(pd.read_csv(file, chunksize=chunk_size, low_memory=False)):
        chunk.columns = chunk.columns.str.strip()
        valid_indices = chunk.index.intersection(final_indices_to_extract)
        
        if len(valid_indices) > 0:
            file_sampled_chunks.append(chunk.loc[valid_indices])

    # lưu ra file cho từng loại tấn công
    if file_sampled_chunks:
        df_final = pd.concat(file_sampled_chunks)
        
        # sampled_ + filename
        output_file_path = os.path.join(output_dir, f"sampled_{file_name}")
        df_final.to_csv(output_file_path, index=False)
        
        print(f" Đã lưu: {output_file_path}")
        print(f" Kích thước: {df_final.shape[0]} dòng, {df_final.shape[1]} cột.")
        if 'Label' in df_final.columns:
            print(f" Tỷ lệ nhãn thực tế:\n{df_final['Label'].value_counts(normalize=True) * 100}")
            
        # Xóa biến để giải phóng RAM cho vòng lặp tiếp theo
        del df_final
        del file_sampled_chunks
    else:
        print("[ERROR] Không rút trích được dữ liệu nào.")


--- ĐANG XỬ LÝ FILE: DrDoS_DNS.csv ---
Hoàn tất Pass 1. Tổng số dòng: 5074413 | Số nhãn 0 (BENIGN): 3402
Số lượng nhãn 1 (DDoS) cần lấy xung quanh nhãn 0: 7938
Rút trích dữ liệu từ file...
 Đã lưu: D:\detection_ddos\data\CSV-01-12\training\cleaned\sampled_DrDoS_DNS.csv
 Kích thước: 11341 dòng, 88 cột.
 Tỷ lệ nhãn thực tế:
Label
DrDoS_DNS    70.002645
BENIGN       29.997355
Name: proportion, dtype: float64

--- ĐANG XỬ LÝ FILE: DrDoS_LDAP.csv ---
Hoàn tất Pass 1. Tổng số dòng: 2181542 | Số nhãn 0 (BENIGN): 1612
Số lượng nhãn 1 (DDoS) cần lấy xung quanh nhãn 0: 3761
Rút trích dữ liệu từ file...
 Đã lưu: D:\detection_ddos\data\CSV-01-12\training\cleaned\sampled_DrDoS_LDAP.csv
 Kích thước: 5373 dòng, 88 cột.
 Tỷ lệ nhãn thực tế:
Label
DrDoS_LDAP    69.998139
BENIGN        30.001861
Name: proportion, dtype: float64

--- ĐANG XỬ LÝ FILE: DrDoS_MSSQL.csv ---
Hoàn tất Pass 1. Tổng số dòng: 4524498 | Số nhãn 0 (BENIGN): 2006
Số lượng nhãn 1 (DDoS) cần lấy xung quanh nhãn 0: 4680
Rút trích dữ l

xử lý nhãn và các cột mang giá trị không quan trọng

In [8]:
import pandas as pd
import numpy as np
import os
import glob

cleaned_dir = r"D:\detection_ddos\data\CSV-01-12\training\cleaned"
sampled_files = glob.glob(os.path.join(cleaned_dir, "sampled_*.csv"))

if not sampled_files:
    print(f"Không tìm thấy file sampled_*.csv nào trong {cleaned_dir}")
else:
    print(f"Bắt đầu gộp {len(sampled_files)} file mẫu để dọn dẹp và xử lý nhãn...\n")
    
    #  gộp file
    df_list = []
    for file in sampled_files:
        df_temp = pd.read_csv(file, low_memory=False)
        df_list.append(df_temp)
        
    df_combined = pd.concat(df_list, ignore_index=True)
    print(f" -> Kích thước sau khi gộp tổng: {df_combined.shape}")

    # Xóa các cột không mang giá trị quan trọng trong quá trình huấn luyện
    identifiers_to_drop = [
        'Flow ID', 'Source IP', 'Src IP', 'Src Port', 
        'Destination IP', 'Dst IP', 'Dst Port', 
        'Timestamp', 'SimillarHTTP', 'Inbound', 'Unnamed: 0'
    ]
    cols_to_drop = [c for c in identifiers_to_drop if c in df_combined.columns]
    df_combined.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    #Xử lý Infinity và biến thành NaN, sau đó xóa các dòng NaN
    df_combined.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_combined.dropna(inplace=True)
    
    # tính phương sai và xóa các cột chỉ có 1 giá trị
    nunique = df_combined.nunique()
    cols_to_drop_var = nunique[nunique <= 1].index.tolist()
    
    if 'Label' in cols_to_drop_var:
        cols_to_drop_var.remove('Label')
        
    df_combined.drop(columns=cols_to_drop_var, inplace=True)
    
    # Xử lý nhãn(label)
    print("\nĐang tạo các cột nhãn phân loại (Binary & Multiclass)...")
    
    # Chuẩn hóa tên Label gốc 
    df_combined['Label'] = df_combined['Label'].astype(str).str.strip().str.upper()
    
    # Cột 1: Label_Binary (0: Bình thường, 1: Tấn công) 
    df_combined['Label_Binary'] = df_combined['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
    
    # Cột 2: Label_Multiclass (0: Bình thường, 1->N: Tấn công) 
    # Lấy danh sách các loại tấn công (bỏ BENIGN ra riêng) và sắp xếp alphabet
    attack_types = sorted([label for label in df_combined['Label'].unique() if label != 'BENIGN'])
    
    # Tạo bảng ánh xạ tĩnh: BENIGN luôn là số 0, các loại khác được đánh số từ 1 trở đi
    mapping_dict = {'BENIGN': 0}
    for i, attack in enumerate(attack_types, start=1):
        mapping_dict[attack] = i
        
    df_combined['Label_Multiclass'] = df_combined['Label'].map(mapping_dict)
    
    # Cột 3: Đổi tên cột gốc thành Label_String để giữ lại text 
    df_combined.rename(columns={'Label': 'Label_String'}, inplace=True)
    
    # Ghi ra một file tổng đã sạch sẽ
    output_file = os.path.join(cleaned_dir, "final_cleaned_combined.csv")
    df_combined.to_csv(output_file, index=False)
    
    print("\n" + "=" * 50)
    print("HOÀN TẤT QUÁ TRÌNH DỌN RÁC VÀ ĐỒNG NHẤT DỮ LIỆU!")
    print(f" -> Kích thước cuối cùng: {df_combined.shape[0]} dòng, {df_combined.shape[1]} cột")
    print(f" -> Đã xóa tổng cộng {len(cols_to_drop) + len(cols_to_drop_var)} cột rác/nhiễu.")
    print(f" -> File sẵn sàng cho các bước tiếp theo được lưu tại: {output_file}\n")
    
    print("--- BẢNG MÃ HÓA NHÃN (MULTICLASS MAPPING) ---")
    for k, v in mapping_dict.items():
        print(f" MÃ {v} : {k}")
        
    print("\n--- THỐNG KÊ TỶ LỆ DỮ LIỆU (BINARY) ---")
    print(df_combined['Label_Binary'].value_counts(normalize=True) * 100)

Bắt đầu gộp 11 file mẫu để dọn dẹp và xử lý nhãn...

 -> Kích thước sau khi gộp tổng: (189542, 88)

Đang tạo các cột nhãn phân loại (Binary & Multiclass)...

HOÀN TẤT QUÁ TRÌNH DỌN RÁC VÀ ĐỒNG NHẤT DỮ LIỆU!
 -> Kích thước cuối cùng: 187005 dòng, 71 cột
 -> Đã xóa tổng cộng 19 cột rác/nhiễu.
 -> File sẵn sàng cho các bước tiếp theo được lưu tại: D:\detection_ddos\data\CSV-01-12\training\cleaned\final_cleaned_combined.csv

--- BẢNG MÃ HÓA NHÃN (MULTICLASS MAPPING) ---
 MÃ 0 : BENIGN
 MÃ 1 : DRDOS_DNS
 MÃ 2 : DRDOS_LDAP
 MÃ 3 : DRDOS_MSSQL
 MÃ 4 : DRDOS_NETBIOS
 MÃ 5 : DRDOS_NTP
 MÃ 6 : DRDOS_SNMP
 MÃ 7 : DRDOS_SSDP
 MÃ 8 : DRDOS_UDP
 MÃ 9 : SYN
 MÃ 10 : TFTP
 MÃ 11 : UDP-LAG
 MÃ 12 : WEBDDOS

--- THỐNG KÊ TỶ LỆ DỮ LIỆU (BINARY) ---
Label_Binary
1    69.82701
0    30.17299
Name: proportion, dtype: float64


ma trận tương quan

In [ ]:
import pandas as pd
import numpy as np
import os

cleaned_dir = r"D:\detection_ddos\data\CSV-01-12\training\cleaned"
input_file = os.path.join(cleaned_dir, "final_cleaned_combined.csv")
output_file = os.path.join(cleaned_dir, "final_reduced_combined.csv")

print("Đang đọc dữ liệu để tính Ma trận tương quan...")
df = pd.read_csv(input_file, low_memory=False)

# Tách riêng các cột Label để không đưa vào tính toán tương quan
label_cols = ['Label_String', 'Label_Binary', 'Label_Multiclass']
labels_df = df[label_cols]
features_df = df.drop(columns=label_cols)

print(f" -> Kích thước đặc trưng ban đầu: {features_df.shape}")
print("Đang tính toán Ma trận tương quan...")

# Tính ma trận tương quan tuyệt đối (Absolute Correlation Matrix)
corr_matrix = features_df.corr().abs()

# Lấy nửa trên (upper triangle) của ma trận để tránh xét các cặp trùng lặp
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Tìm các cột có độ tương quan > 0.90
THRESHOLD = 0.90
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > THRESHOLD)]

print(f"\nTìm thấy {len(to_drop)} cột có độ tương quan > {THRESHOLD}.")
if len(to_drop) > 0:
    print(f" -> Danh sách một số cột sẽ bị xóa: {to_drop[:10]} ...")

# Xóa các cột dư thừa khỏi tập đặc trưng
features_df.drop(columns=to_drop, inplace=True)

# Ghép lại Label với tập đặc trưng đã được tối ưu
df_final = pd.concat([features_df, labels_df], axis=1)

# Ghi ra file mới
df_final.to_csv(output_file, index=False)

print("\n" + "=" * 50)
print("HOÀN TẤT LỌC ĐẶC TRƯNG BẰNG MA TRẬN TƯƠNG QUAN!")
print(f" -> Kích thước dữ liệu cuối cùng: {df_final.shape[0]} dòng, {df_final.shape[1]} cột.")
print(f" -> Đã lưu file chuẩn bị cho bước Scale tại: {output_file}")

Đang đọc dữ liệu để tính Ma trận tương quan...
 -> Kích thước đặc trưng ban đầu: (187005, 68)
Đang tính toán Ma trận tương quan...

Tìm thấy 25 cột có độ tương quan > 0.9.
 -> Danh sách một số cột sẽ bị xóa: ['Fwd Packet Length Mean', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Std', 'Bwd IAT Max', 'Fwd Packets/s', 'Min Packet Length'] ...

HOÀN TẤT LỌC ĐẶC TRƯNG BẰNG MA TRẬN TƯƠNG QUAN!
 -> Kích thước dữ liệu cuối cùng: 187005 dòng, 46 cột.
 -> Đã lưu file chuẩn bị cho bước Scale tại: D:\detection_ddos\data\CSV-01-12\training\cleaned\final_reduced_combined.csv
